# TRNG HRNG / DRNG / PTRNG

These cells use **recorded hardware evidence** by default. They do not program a board.
Use the configured Python (SCU35 PMC) kernel. Fresh hardware commands are shown in Markdown and must be invoked explicitly after configuring your exact SCU35 and recovery image.
Current PL timing uses AXI Timer; native PMC timing uses `mcycle`. JTAG/UART are outside operation timing.

The PMC peripheral name is PMCL_TRNG; the public driver is XTrngpsx. Driver setup includes self-tests and instantiation. Public-seeded DRNG is diagnostic, not production entropy. Python OS randomness is a different service; bytes must not match and smoke tests are not entropy certification.

In [1]:
from pathlib import Path
import hashlib, json, sys, time
import pandas as pd
# Works from the notebook directory, lab root, or parent repository.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "scu35") if (p / "lab.py").is_file()), None)
assert ROOT is not None, "Launch Jupyter inside this checkout or its scu35 directory"
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "host"))
from compare import parse_capture
from lab import config
capture = parse_capture(ROOT / "evidence/timer/uart.txt", "7d1bf77bc454dd1f")
rows = pd.DataFrame(capture["rows"])
print("Recorded board capture: 31 complete rows; current Python:", sys.version.split()[0])


Recorded board capture: 31 complete rows; current Python: 3.11.9


In [2]:
import os
display(rows[rows.algorithm.str.contains("RNG")])
display(pd.DataFrame({k:v for k,v in capture["features"].items() if k.startswith("TRNG")}).T)
start = time.perf_counter_ns()
for _ in range(1000): random_bytes = os.urandom(32)
print("Python OS RNG mean us:", (time.perf_counter_ns()-start)/1000/1000)
assert len(random_bytes) == 32

,algorithm,case,size_bytes,repeats,copy_cycles,setup_cycles,api_cycles,check_cycles,scu35_api_mean_us,scu35_full_mean_us,scu35_operation_mean_us,scu35_min_us,scu35_max_us,status
28,TRNG-HRNG,generate,32,20,0,0,3247031,0,1623.5155,1623.5155,1623.5155,6.76,32341.87,0
29,TRNG-DRNG,generate,32,20,0,0,13901,0,6.9505,6.9505,6.9505,6.66,12.47,0
30,TRNG-PTRNG,generate,32,20,0,0,17229270,0,8614.6350,8614.6350,8614.6350,8607.55,8622.69,0


,repeats,cycles,mean_us,status
TRNG-HRNG-SETUP,1.0,3628912.0,36289.12,0.0
TRNG-DRNG-SETUP,1.0,3678294.0,36782.94,0.0
TRNG-PTRNG-SETUP,1.0,3625949.0,36259.49,0.0


Python OS RNG mean us: 0.2515


## Fresh hardware: explicit step

Configure `config.json` first. Run `python lab.py hardware --pdi /path/to/build/demo.pdi` from the lab directory for the full runtime suite, or the native/boot command above for that path. Save logs, verify recovery and compare results before discussing speed. See [audit checkpoints](../docs/audit.md).